# Week 4 Meeting 1 — RAG over physics papers

A minimal retrieval-augmented-generation pipeline: PDFs → text chunks → sentence embeddings → FAISS index → Claude answers questions using only the retrieved chunks.

**How to use**

1. Put any PDFs you want to query into the `pdfs/` folder next to this notebook (optional — you can also pull papers from arXiv by keyword in the *Build the corpus* cell).
2. Put your Anthropic key in a file called `.env` next to this notebook: `ANTHROPIC_API_KEY=sk-ant-...` (never paste the key into a cell).
3. *Kernel → Restart Kernel and Run All Cells*.
4. Ask questions at the bottom with `ask("...")`.

Everything is cached: PDFs are downloaded once, and the embedding model is loaded once per kernel. Embedding runs on CPU at roughly 10 chunks per second (a 300-page textbook ≈ 1–2 minutes, once). Each `ask()` sends about 2 000 input tokens to Claude Haiku — a fraction of a cent.

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv

load_dotenv(Path(".env"))  # reads .env in this folder, if there is one

if not os.environ.get("ANTHROPIC_API_KEY", "").startswith("sk-ant-"):
    # No .env (e.g. first run on a JupyterHub): ask once with a masked prompt. The key lives only in
    # this kernel's memory - it is never written into the notebook, so the notebook stays safe to share.
    from getpass import getpass
    os.environ["ANTHROPIC_API_KEY"] = getpass("Paste your Anthropic API key (input is hidden): ").strip()
if not os.environ.get("ANTHROPIC_API_KEY", "").startswith("sk-ant-"):
    raise RuntimeError("No valid ANTHROPIC_API_KEY (should start with sk-ant-). Re-run this cell, or put the key in a .env file next to the notebook.")
print("API key loaded.")

PDF_DIR = Path("pdfs")            # drop your own PDFs here
ARXIV_DIR = Path("arxiv_files")   # arXiv downloads are cached here
PDF_DIR.mkdir(exist_ok=True)
ARXIV_DIR.mkdir(exist_ok=True)

CLAUDE_MODEL = "claude-haiku-4-5-20251001"   # fast + cheap. For harder questions try "claude-sonnet-4-6" (≈10x the price)
EMBEDDING_MODEL = "all-MiniLM-L6-v2"          # 384-dim sentence embeddings, runs on CPU

API key loaded.


## 1. Build the corpus

Two sources, same output format: a list of chunk dicts `{"id", "title", "text_content", "url", "page"}`.

- `load_local_pdfs()` reads every PDF in `pdfs/`.
- `fetch_papers(query, n)` searches arXiv and downloads the PDFs (cached in `arxiv_files/`); if arXiv's API is rate-limiting (HTTP 429 — routine on a shared JupyterHub where the whole class queries at once) it finds arXiv papers through **OpenAlex** instead and downloads from arXiv's PDF server, which is not throttled.
- `load_fallback_corpus()` loads the 30 abstracts the course ships for offline use — a last resort, since abstracts give shallow answers.

Chunks are ~150 words with a 30-word overlap over the whole document, each tagged with the page(s) it came from, so answers can cite *paper and page*. (The skeleton split on blank lines, which `pypdf` almost never emits — that produced one giant chunk per paper. 150 words also keeps every chunk inside the embedding model's 256-token window; longer chunks are silently truncated by the embedder, so their tails become unsearchable.)

Limitation to keep in mind: `pypdf` reads the PDF *text layer*. Typeset equations usually come out blank or garbled ("where ξ is , built on ,"), and scanned/image-only PDFs yield nothing. Prose, headings and captions are fine, and that is what retrieval keys on.

In [2]:
import logging
import re

import arxiv
import pypdf

# pypdf logs a long "fontTools is required..." warning per font for LaTeX-generated PDFs (i.e. every arXiv
# paper). Text extraction is unaffected (install `fonttools` to make it go away for good), so silence it.
logging.getLogger("pypdf").setLevel(logging.ERROR)

HTTP_HEADERS = {"User-Agent": "rag-skeleton/1.0 (PHYS 7440 student notebook)"}


def chunk_words(words, size: int = 150, overlap: int = 30, min_words: int = 12):
    """Yield (start, end) word-index windows of `size` with `overlap`; the final window may be
    short but never shorter than `min_words` (so a 30-word title page is still indexed).

    150 words is chosen to stay under the embedding model's 256-token limit even for
    equation-dense text (which tokenises at ~1.5 tokens/word); anything past that limit
    would be silently ignored by the embedder, i.e. invisible to retrieval."""
    step = max(1, size - overlap)
    n = len(words)
    start = 0
    while start < n:
        end = min(start + size, n)
        if end - start >= min_words:
            yield start, end
        if end == n:
            break
        start += step


def pdf_to_chunks(pdf_path: Path, title: str | None = None, url: str | None = None):
    """Extract the whole document, then chunk it with a word->page map, so a paragraph that
    runs across a page break is kept together and every chunk still knows its page."""
    reader = pypdf.PdfReader(str(pdf_path))
    if title is None:
        meta_title = (reader.metadata or {}).get("/Title") if reader.metadata else None
        meta_title = str(meta_title).strip() if meta_title else ""
        # PDF metadata titles are often junk ('untitled', 'Microsoft Word - Document1'); when two
        # different files would collide on the same title, keep the filename so citations stay unambiguous
        generic = not meta_title or len(meta_title) < 8 or meta_title.lower() in {"untitled", "document", "title"} \
            or meta_title.lower().startswith(("microsoft word", "untitled", "document"))
        title = pdf_path.stem if generic else f"{meta_title} [{pdf_path.name}]"

    words, pages = [], []
    for page_no, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        text = re.sub(r"-\n(?=[a-z])", "", text)   # re-join words hyphenated across lines
        page_words = text.split()
        words.extend(page_words)
        pages.extend([page_no] * len(page_words))

    stem = re.sub(r"[^A-Za-z0-9._-]+", "_", pdf_path.stem).strip("_") or "doc"
    chunks = []
    for k, (a, b) in enumerate(chunk_words(words)):
        first, last = pages[a], pages[b - 1]
        chunks.append({
            "id": f"{stem}_c{k}",
            "title": title,
            "text_content": " ".join(words[a:b]),
            "url": url or str(pdf_path),
            "page": first if first == last else f"{first}-{last}",
        })
    return chunks


def load_local_pdfs(folder: Path = PDF_DIR):
    """Chunk every PDF in `folder` (any case of .pdf). Unreadable files are reported and skipped."""
    corpus = []
    pdfs = sorted(p for p in folder.iterdir() if p.is_file() and p.suffix.lower() == ".pdf")
    if not pdfs:
        print(f"  (no PDFs in {folder.resolve()})")
    for pdf in pdfs:
        try:
            found = pdf_to_chunks(pdf)
            corpus.extend(found)
            print(f"  {pdf.name}: {len(found)} chunks" + ("  <- no text layer (scanned?)" if not found else ""))
        except pypdf.errors.FileNotDecryptedError:
            print(f"  skipping {pdf.name}: password-protected - open it in a PDF viewer and 'print to PDF' to make an unlocked copy")
        except Exception as e:
            print(f"  skipping {pdf.name}: {type(e).__name__}: {e}")
    return corpus


def fetch_arxiv_pdfs(query: str, max_results: int = 3, timeout: int = 60):
    """Search arXiv's API, download each PDF once (cached), and chunk it.

    Fix vs. skeleton: ``download_pdf`` is not a ``Client`` method (older ``arxiv`` versions
    have it on ``Result``; v4 removed it entirely). Downloading ``r.pdf_url`` directly works
    on every version. If the API is unreachable or rate-limited (HTTP 429 - common on a
    shared JupyterHub where a whole class hits arXiv at once) this returns [] with a message
    instead of crashing; ``fetch_papers`` below then searches via OpenAlex.
    """
    import requests

    corpus = []
    print(f"arXiv search {query!r} (up to {max_results} papers)")
    try:
        client = arxiv.Client(num_retries=2, delay_seconds=5)
        results = list(client.results(arxiv.Search(query=query, max_results=max_results)))
    except Exception as e:
        hint = " - arXiv is rate-limiting this network (HTTP 429); trying OpenAlex next" if "429" in str(e) else ""
        print(f"  arXiv API unavailable ({type(e).__name__}){hint}")
        return corpus
    for r in results:
        paper_id = r.get_short_id().replace("/", "_")
        pdf_path = ARXIV_DIR / f"{paper_id}.pdf"
        try:
            if not pdf_path.exists():
                resp = requests.get(r.pdf_url, timeout=timeout, headers=HTTP_HEADERS)
                resp.raise_for_status()
                pdf_path.write_bytes(resp.content)
            found = pdf_to_chunks(pdf_path, title=" ".join(r.title.split()), url=r.entry_id)
            corpus.extend(found)
            print(f"  {paper_id}: {len(found)} chunks - {r.title[:70]}")
        except Exception as e:
            print(f"  skipping {paper_id}: {e}")
    return corpus


ARXIV_SOURCE_ID = "S4306400194"   # OpenAlex's id for arXiv


def fetch_openalex_arxiv_pdfs(query: str, max_results: int = 3, timeout: int = 60):
    """Fallback search that avoids arXiv's rate-limited *API* entirely: ask OpenAlex (free, no
    key) for papers that have an arXiv version, then download from arXiv's *PDF server*, which
    is a different service and keeps answering 200 while export.arxiv.org returns 429."""
    import requests

    corpus = []
    print(f"OpenAlex search {query!r} for arXiv papers (up to {max_results})")
    params = {"search": query, "filter": f"locations.source.id:{ARXIV_SOURCE_ID}", "per-page": max_results * 2,
              "select": "id,title,locations", "mailto": "student@myumanitoba.ca"}
    try:
        works = requests.get("https://api.openalex.org/works", params=params, timeout=timeout, headers=HTTP_HEADERS).json()["results"]
    except Exception as e:
        print(f"  OpenAlex unavailable ({type(e).__name__}: {e})")
        return corpus
    for w in works:
        if len({c["title"] for c in corpus}) >= max_results:
            break
        arxiv_locs = [l for l in w.get("locations", []) if ((l.get("source") or {}).get("id") or "").endswith(ARXIV_SOURCE_ID)]
        if not arxiv_locs or not arxiv_locs[0].get("pdf_url"):
            continue
        pdf_url = arxiv_locs[0]["pdf_url"]
        paper_id = pdf_url.rsplit("/", 1)[-1].replace("/", "_")
        title = " ".join((w.get("title") or paper_id).split())
        pdf_path = ARXIV_DIR / f"{paper_id}.pdf"
        try:
            if not pdf_path.exists():
                resp = requests.get(pdf_url, timeout=timeout, headers=HTTP_HEADERS)
                resp.raise_for_status()
                if not resp.content.startswith(b"%PDF"):
                    raise ValueError("did not receive a PDF")
                pdf_path.write_bytes(resp.content)
            found = pdf_to_chunks(pdf_path, title=title, url=arxiv_locs[0].get("landing_page_url") or pdf_url)
            corpus.extend(found)
            print(f"  {paper_id}: {len(found)} chunks - {title[:70]}")
        except Exception as e:
            print(f"  skipping {paper_id}: {e}")
    return corpus


def fetch_papers(query: str, max_results: int = 3):
    """Try arXiv's API first; if it is rate-limited or down, find arXiv papers via OpenAlex instead."""
    corpus = fetch_arxiv_pdfs(query, max_results)
    if not corpus:
        corpus = fetch_openalex_arxiv_pdfs(query, max_results)
    return corpus


def load_fallback_corpus(path: Path = Path("fallback_corpus.json")):
    """The course repo ships 30 abstracts (docs/exercises/week04/fallback_corpus.json) for exactly
    the case where arXiv is unreachable. Downloaded once if not already next to the notebook.
    These are abstracts, not full papers, so answers will be correspondingly shallow."""
    import json
    import requests

    if not path.exists():
        url = "https://raw.githubusercontent.com/wdconinc/ai-in-physics-research/main/docs/exercises/week04/fallback_corpus.json"
        try:
            path.write_bytes(requests.get(url, timeout=60, headers=HTTP_HEADERS).content)
        except Exception as e:
            print(f"  could not fetch fallback corpus: {e}")
            return []
    papers = json.loads(path.read_text())
    corpus = [{"id": f"fallback_{p['id']}", "title": p["title"], "text_content": p["abstract"],
               "url": p.get("url", ""), "page": "abstract"} for p in papers]
    print(f"  fallback corpus: {len(corpus)} abstracts from {path}")
    return corpus

## 2. Embed, index, retrieve

In [3]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(EMBEDDING_MODEL)   # loaded once per kernel


def embed_texts(texts):
    """Dense embeddings as float32, L2-normalised so inner product == cosine similarity."""
    if isinstance(texts, str):
        texts = [texts]
    texts = list(texts)
    vecs = embedding_model.encode(texts, convert_to_numpy=True, show_progress_bar=len(texts) > 100, batch_size=64)
    vecs = np.ascontiguousarray(vecs, dtype="float32")
    faiss.normalize_L2(vecs)
    return vecs


def build_index(embeddings):
    """Exact cosine-similarity index."""
    if embeddings.ndim != 2 or embeddings.shape[0] == 0:
        raise ValueError("No chunks to index - add PDFs to pdfs/ or fetch some from arXiv first.")
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)
    return index


def retrieve(query: str, corpus, index, k: int = 5, max_per_doc: int | None = None):
    """Top-k chunks for a query as (chunk, score) pairs.

    - Duplicate texts (same PDF added twice, repeated running headers) are skipped.
    - `max_per_doc` caps how many chunks one document may contribute, so a question that
      spans papers is not answered from whichever paper happens to use the question's words
      most often. Default: no cap for a single-document corpus, otherwise ceil(k / n_docs) + 1.
    """
    if not corpus:
        return []
    n_docs = len({c["title"] for c in corpus})
    if max_per_doc is None:
        max_per_doc = k if n_docs == 1 else -(-k // n_docs) + 1
    scores, ids = index.search(embed_texts(query), min(10 * k, len(corpus)))
    out, seen, per_doc = [], set(), {}
    for i, s in zip(ids[0], scores[0]):
        if i == -1:
            continue
        chunk = corpus[i]
        text, doc = chunk["text_content"], chunk["title"]
        if text in seen or per_doc.get(doc, 0) >= max_per_doc:
            continue
        seen.add(text)
        per_doc[doc] = per_doc.get(doc, 0) + 1
        out.append((chunk, float(s)))
        if len(out) == k:
            break
    return out

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## 3. Answer with Claude

In [4]:
from anthropic import Anthropic

claude = Anthropic()   # picks up ANTHROPIC_API_KEY from the environment loaded above


def answer_with_rag(question: str, retrieved, model: str = CLAUDE_MODEL, max_tokens: int = 1024):
    """Send the retrieved chunks as context and return Claude's answer as a string.

    Fix vs. skeleton: ``message.content`` is a *list* of content blocks, so the text is
    ``message.content[0].text`` (joined over all text blocks here), not ``message.content.text``.
    """
    if not retrieved:
        return "Nothing retrieved - the corpus is empty or the index was not built."
    context = "\n\n---\n\n".join(
        f"[{n}] Source: {item['title']} (page {item['page']})\n{item['text_content']}"
        for n, (item, _score) in enumerate(retrieved, start=1)
    )
    system_prompt = (
        "You are an expert scientific assistant. Answer using ONLY the numbered context excerpts. "
        "Cite them inline as [n] and name the source paper. If the context does not contain the answer, say so."
    )
    message = claude.messages.create(
        model=model,
        max_tokens=max_tokens,
        system=system_prompt,
        messages=[{"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}],
    )
    return "".join(block.text for block in message.content if getattr(block, "type", "") == "text")


def ask(question: str, k: int = 6, show_sources: bool = True):
    """One-call helper: retrieve top-k chunks for `question`, then answer with Claude."""
    hits = retrieve(question, corpus, index, k=k)
    if show_sources:
        print("Sources used:")
        for item, score in hits:
            print(f"  {score:.2f}  {item['title'][:60]}  (p.{item['page']})")
        print()
    answer = answer_with_rag(question, hits)
    print(answer)
    return answer

## 4. Build the corpus and index

Edit the two lines below: add PDFs to `pdfs/`, and/or change the arXiv query.

In [5]:
QUERY = "Electrotaxis"          # used only if pdfs/ is empty

corpus = []
print("Local PDFs:")
corpus += load_local_pdfs()                       # 1. everything in pdfs/
if not corpus:
    corpus += fetch_papers(QUERY, max_results=3)  # 2. arXiv, then OpenAlex if arXiv is rate-limited
if not corpus:
    corpus += load_fallback_corpus()              # 3. the course's 30 offline abstracts
# To mix sources, call fetch_papers(...) / load_fallback_corpus() unconditionally and += them.

print(f"\n{len(corpus)} chunks from {len({c['title'] for c in corpus})} documents")
embeddings = embed_texts([c["text_content"] for c in corpus])
index = build_index(embeddings)
print("index ready:", index.ntotal, "vectors of dim", index.d)

Local PDFs:
  (no PDFs in /home/jupyter/PHYS 7440 Gen AI/Week 04/pdfs)
arXiv search 'Electrotaxis' (up to 3 papers)
  2402.08700v1: 86 chunks - Optimal control of collective electrotaxis in epithelial monolayers
  2401.14376v1: 52 chunks - Electrotaxis of self-propelling artificial swimmers in microchannels
  0710.0523v1: 15 chunks - Fluctuation theorem applied to Dictyostelium discoideum system

153 chunks from 3 documents


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

index ready: 153 vectors of dim 384


## 5. Ask questions

`ask()` prints the sources it used and Claude's answer. Change `k` to give Claude more or fewer excerpts.

In [6]:
_ = ask(" what the methodology is about in one paragraph, and list its main results or sections.")

Sources used:
  0.13  Optimal control of collective electrotaxis in epithelial mon  (p.9-10)
  0.12  Optimal control of collective electrotaxis in epithelial mon  (p.9)
  0.11  Optimal control of collective electrotaxis in epithelial mon  (p.24-25)
  0.08  Fluctuation theorem applied to Dictyostelium discoideum syst  (p.1)
  0.06  Fluctuation theorem applied to Dictyostelium discoideum syst  (p.2)
  0.06  Electrotaxis of self-propelling artificial swimmers in micro  (p.6)

# Methodology and Main Results

Based on the provided context, I can describe the methodology from the "Optimal control of collective electrotaxis in epithelial monolayers" paper [2]:

**Methodology:** The study employs a Bayesian inference approach using Markov Chain Monte Carlo (MCMC) procedures to identify model parameters describing collective electrotaxis in epithelial monolayers. The researchers fit a mathematical model to experimental data and then compute posterior predictive intervals (95% confidence interva

In [7]:
_ = ask("I think you should mention more for the methods")

Sources used:
  0.21  Optimal control of collective electrotaxis in epithelial mon  (p.10-11)
  0.18  Optimal control of collective electrotaxis in epithelial mon  (p.10)
  0.16  Optimal control of collective electrotaxis in epithelial mon  (p.16)
  0.11  Fluctuation theorem applied to Dictyostelium discoideum syst  (p.1)
  0.09  Electrotaxis of self-propelling artificial swimmers in micro  (p.6)
  0.09  Fluctuation theorem applied to Dictyostelium discoideum syst  (p.1)

I appreciate your feedback, but I need to clarify my role. I'm instructed to answer questions using ONLY the numbered context excerpts provided to you. 

The context you've given me contains limited methodological detail. The most substantial methods information available is in the provided excerpts, which includes:

- [1] and [2] discuss the optimal control framework and the adaptation-excitation model used to describe electrotaxis
- [3] mentions implementation details: "a damped Newton method with an affineinvariant

In [8]:
# Ask anything about your PDFs - a few patterns:
# _ = ask("What does the author say about <topic>?")
# _ = ask("Compare how the different documents treat <topic>.")
# _ = ask("What limitations or open questions are mentioned?")
# _ = ask("Give me a 5-bullet study summary of <document title>.", k=10)

# To inspect retrieval without calling Claude:
# for item, score in retrieve("<your query>", corpus, index, k=5):
#     print(f"{score:.3f} | p.{item['page']} | {item['text_content'][:120]}...")

In [9]:
pip install arxiv pypdf faiss-cpu sentence_transformers

Note: you may need to restart the kernel to use updated packages.
